# A7a · Qwen3.8-Flash-Next (HC + QSA indexer + n-gram PLE + MoE) 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/A7a_qwen38_flash_next.ipynb)

원본: transformers `qwen4_exp` (`Qwen4ExpTextModel`, 텍스트 부분만). 내 구현: `miniarch/models/qwen38_flash_next.py`.
Gated DeltaNet과 partial RoPE는 A6a(`qwen36.py`) 부품을 그대로 씁니다.

부품
- Qwen3.6 하이브리드 뼈대: Gated DeltaNet 3층 : Gated Attention 1층 (DeltaNet 출력 게이트는 sigmoid)
- Hyper-Connections: residual을 줄기 4개로 들고 다니며, 층 앞에서 저랭크 sigmoid 가중치로 줄기를 합쳐 입력을 만들고 출력은 줄기마다 다른 세기로 주입합니다. 마지막에 같은 방식으로 하나로 합칩니다 (final norm 없음).
- QSA indexer: 지난 토큰을 4개씩 묶어 평균 key로 요약하고, 점수 상위 묶음과 아직 덜 찬 꼬리 묶음의 토큰만 attention합니다.
- PLE: 2-gram, 3-gram을 해시해서 표에서 임베딩을 꺼내 줄기별 게이트로 더합니다. EOS에서 n-gram 문맥이 끊깁니다.
- MoE: softmax top-k 라우터 + sigmoid 게이트 shared expert

축소 config: 4층 = [선형, 선형(PLE), 선형(PLE), indexed], hidden 64, 줄기 4개, 시퀀스 80.
- 시퀀스 80 → DeltaNet chunk(64) 2개 + padding 경로
- indexer 묶음 4토큰, budget 8(묶음 2개) → 묶음 0개, 2개 이하(전부 선택), 2개 초과(top-k 선택), 꼬리 묶음 경로가 모두 쓰입니다.
- PLE 층 2개 → 층마다 다른 해시 표 크기와 곱셈 상수. 입력에 EOS를 넣어(연속 EOS 포함) n-gram 끊김 경로까지 검사합니다.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. causal 누수 → 5. 단계 전용 자체 점검 → 6. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip -q install -U -r requirements-colab.txt
# Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
!pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("A7a")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('Qwen/Qwen3.8-Flash-Next', 'layers\\.(1|3)\\.(?!.*ngram_embedding)', subfolder='')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## causal 누수

뒤쪽 토큰을 바꿨을 때 앞쪽 출력이 변하면 미래 정보가 새고 있다는 뜻입니다.

In [ ]:
import importlib
V.set_deterministic(0)
mine = importlib.import_module(S.CASE.mine_module).verify_pair(S.ref()[0])["mine"].to(dev).eval()
r = V.causal_leak_test(lambda x: mine(x).last_hidden_state, vocab_size=V.cases.VOCAB, seq_len=48, device=dev, atol=1e-4)
report("내 구현 " + r.name, r.ok, r.max_abs)

## 단계 전용 자체 점검

### QSA indexer 선택 · attention map 확인

- `select_probe`: 질의마다 허용된 토큰. 개수 = min(완성 묶음 수, 2)·4 + 꼬리 길이여야 하고, 미래 토큰은 없어야 합니다.
- `attn_probs`: 허용되지 않은 칸의 확률은 정확히 0, 행 합은 1이어야 합니다.

In [ ]:
from miniarch.models import qwen38_flash_next as M
ref, inp = S.ref()
m = M.verify_pair(ref)["mine"].to(dev).eval()
L = m.config.layer_types.index(M.INDEXED)
pre = f"layers.{L}.self_attn"
x = inp["input_ids"].to(dev)
with V.capture(m, [pre + ".indexer.select_probe", pre + ".attn_probs"]) as st, torch.no_grad():
    m(x)
sel, p = st[pre + ".indexer.select_probe"] > 0.5, st[pre + ".attn_probs"]
T, r, k = x.shape[1], m.config.index_ratio, m.config.index_budget // m.config.index_ratio
t = torch.arange(1, T + 1, device=dev)
expect = torch.minimum(t // r, torch.tensor(k, device=dev)) * r + t % r
cnt_ok = bool((sel.sum(-1) == expect).all())
future = bool(sel.triu(1).any())
outside = p.masked_fill(sel[:, None], 0).abs().max().item()
rowerr = (p.sum(-1) - 1).abs().max().item()
print(f"layer {L}: 선택 개수 일치={cnt_ok}  미래 토큰 선택={future}  선택 밖 확률 최대={outside:.1e}  행합 오차={rowerr:.1e}")
print("질의별 선택 수 (앞 24개):", sel[0].sum(-1)[:24].tolist())
report("indexer 선택 개수 / 인과성", cnt_ok and not future)
report("attention: 선택 밖 확률 0, 행합 1", outside == 0 and rowerr < 1e-5, max(outside, rowerr))

### PLE: EOS 에서 n-gram 문맥이 끊기는지

EOS 앞쪽 토큰을 바꿔도 EOS 다음 위치들의 n-gram 임베딩은 그대로여야 합니다 (문서 경계 너머를 섞지 않음).

In [ ]:
from miniarch.models import qwen38_flash_next as M
ref, _ = S.ref()
m = M.verify_pair(ref)["mine"].to(dev).eval()
eos = m.config.eos_token_id
ple = next(l.ple for l in m.layers if l.ple is not None)
g = torch.Generator().manual_seed(5)
x = torch.randint(2, V.cases.VOCAB, (2, 40), generator=g)
x[:, 20] = eos
x2 = x.clone()
x2[:, :20] = torch.randint(2, V.cases.VOCAB, (2, 20), generator=g)
with torch.no_grad():
    a, b = ple.ple_embedding(x.to(dev)), ple.ple_embedding(x2.to(dev))
after = (a[:, 21:] - b[:, 21:]).abs().max().item()
before = (a[:, :20] - b[:, :20]).abs().max().item()
print(f"EOS 뒤 차이={after:.1e} (0 이어야 함)   EOS 앞 차이={before:.1e} (0 이 아니어야 함)")
report("PLE n-gram EOS 경계", after == 0 and before > 0, after)

### ablation 스위치 · HC hook 확인

스위치를 하나씩 끈 모델이 돌아가고 causal 을 지키는지, 그리고 원본 설정에서 HC 주입 세기(0~2)와 입력 가중치(0~1) 범위를 봅니다.

In [ ]:
import dataclasses
from miniarch.models import qwen38_flash_next as M
ref, inp = S.ref()
base = M.config_from_hf(ref.config)
for sw in ["use_hc", "use_indexer", "use_ple", "use_moe"]:
    V.set_deterministic(0)
    mm = M.Qwen38FlashNext(dataclasses.replace(base, **{sw: False})).to(dev).eval()
    r = V.causal_leak_test(lambda x: mm(x).last_hidden_state, vocab_size=V.cases.VOCAB, seq_len=32, device=dev, atol=1e-4)
    with torch.no_grad():
        ok = bool(torch.isfinite(mm(inp["input_ids"].to(dev)).logits).all())
    report(f"{sw}=False 동작 + causal", r.ok and ok, r.max_abs)

m = M.verify_pair(ref)["mine"].to(dev).eval()
names = [n for n, _ in m.named_modules() if n.endswith(("inject_probe", "mix_probe"))]
with V.capture(m, names) as st, torch.no_grad():
    m(inp["input_ids"].to(dev))
for i in range(m.config.n_layer):
    inj = st[f"layers.{i}.attn_hyper_connection.inject_probe"]
    mix = st[f"layers.{i}.attn_hyper_connection.mix_probe"]
    print(f"layer {i} {m.config.layer_types[i]:<18} 주입 세기 평균(줄기별)={[round(v, 3) for v in inj.mean((0, 1)).tolist()]}  입력 가중치 평균={mix.mean():.3f}")

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/A7a_qwen38_flash_next.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_A7a_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")